# Rapport SQL 

### Import des librairies 

In [10]:
import sqlite3
from random import choice, randint, uniform
from datetime import datetime, timedelta


### Création des bdd et du curseur 

In [11]:
conn = sqlite3.connect(":memory:")
cursor = conn.cursor()

conn.executescript("""
CREATE TABLE customers(
    id INTEGER PRIMARY KEY,
    name TEXT,
    country TEXT
);

CREATE TABLE orders(
    order_id INTEGER PRIMARY KEY,
    customer_id INTEGER,
    order_date DATETIME,
    amount REAL
);
""")

###### Note : on met en mémoire pour ne rien stocker sur le DD 

### Insertion des données 

In [12]:
names = ["Alice", "Bob", "Charlie", "Diana", "Eve", "Frank", "Grace", "Hector", "Ivy", "Jack"]
countries = ["France", "USA", "Germany", "UK", "Spain"]

# Insertion des clients
for _ in range(10):
    name = choice(names)
    country = choice(countries)
    cursor.execute(
        "INSERT INTO customers (name, country) VALUES (?, ?)",
        (name, country)
    )

# Récupérer les IDs des clients générés
cursor.execute("SELECT id FROM customers")
customer_ids = [row[0] for row in cursor.fetchall()]

# Insertion de commandes aléatoires
order_id = 1
for customer_id in customer_ids:
    for _ in range(randint(0,5)):  # chaque client a entre 0 et 5 commandes
        order_date = datetime.now() - timedelta(days=randint(0,90))
        order_date_str = order_date.strftime("%Y-%m-%d %H:%M:%S")
        amount = round(uniform(10,500),2)
        cursor.execute(
            "INSERT INTO orders (order_id, customer_id, order_date, amount) VALUES (?, ?, ?, ?)",
            (order_id, customer_id, order_date_str, amount)
        )
        order_id += 1

conn.commit()


### Affichage test 

In [13]:
cursor.execute("SELECT * FROM customers")
rows = cursor.fetchall()
column_names = [desc[0] for desc in cursor.description]

print('\t'.join(column_names))
for row in rows:
    print('\t'.join(str(x) for x in row))


id	name	country
1	Alice	UK
2	Jack	UK
3	Jack	USA
4	Diana	Spain
5	Diana	France
6	Grace	Germany
7	Hector	France
8	Bob	USA
9	Frank	USA
10	Hector	Germany


### Première tâche : 10 clients les plus actifs 

In [14]:
cursor.execute("""
SELECT c.name, c.id, COUNT(o.order_id) AS num_orders
FROM customers c
JOIN orders o
ON c.id = o.customer_id
GROUP BY c.id, c.name
ORDER BY num_orders DESC
LIMIT 10
""")

rows = cursor.fetchall()
column_names = [desc[0] for desc in cursor.description]

print('\t'.join(column_names))
for row in rows:
    print('\t'.join(str(x) for x in row))


name	id	num_orders
Hector	10	5
Diana	4	4
Grace	6	4
Jack	3	3
Bob	8	3
Jack	2	2
Frank	9	1


### Deuxième tâche : CA par pays 

In [15]:
cursor.execute("""
SELECT c.country, SUM(o.amount) AS total
FROM customers c
JOIN orders o
ON c.id = o.customer_id
GROUP BY c.country
ORDER BY total DESC
""")

rows = cursor.fetchall()
column_names = [desc[0] for desc in cursor.description]

print('\t'.join(column_names))
for row in rows:
    print('\t'.join(str(x) for x in row))


country	total
Germany	2682.61
USA	2075.39
Spain	1718.3
UK	612.96


### Troisième tâche : clients qui n'ont pas passé commande 

In [16]:
cursor.execute("""
SELECT c.name, c.id
FROM customers c
LEFT JOIN orders o
ON c.id = o.customer_id
WHERE o.order_id IS NULL
""")

rows = cursor.fetchall()
column_names = [desc[0] for desc in cursor.description]

print('\t'.join(column_names))
for row in rows:
    print('\t'.join(str(x) for x in row))


name	id
Alice	1
Diana	5
Hector	7


### Fermeture de connexion 

In [17]:
conn.close()
